# Customer Churn Prediction — End-to-End Analysis

This notebook walks through a complete churn-prediction workflow on a Telco-style dataset:

1. **Exploratory Data Analysis** — class distribution, numeric histograms, correlation heatmap
2. **Data cleaning** — dtype fixes, blank `TotalCharges`, duplicate removal, target encoding
3. **Feature engineering** — 25+ derived features (add-on counts, tenure buckets, interactions, CLV proxy)
4. **Class imbalance** — SMOTE oversampling (applied *inside* CV to avoid leakage)
5. **Model comparison** — Logistic Regression, Random Forest, XGBoost, SVM, KNN via 5-fold stratified CV
6. **Hyperparameter tuning** — `GridSearchCV` on the best model (XGBoost)
7. **Evaluation** — confusion matrix, ROC curve, feature importance
8. **Experiment tracking** — MLflow logs every model's params & metrics

> The reusable logic lives in `src/` and is imported here so the notebook stays readable.
> Running `python -m src.train` reproduces all artifacts (model, metrics, figures) headlessly.

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings("ignore")
sys.path.append("..")   # make the `src` and `data` packages importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")

from src.data_loader import load_raw
from src.preprocessing import clean
from src.feature_engineering import add_features
from src.config import CFG, SEED, TARGET
np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
print("Environment ready. Seed =", SEED)

## 1. Load the data

Uses the real IBM/Kaggle Telco CSV if present in `data/`, otherwise a synthetic Telco-style dataset (`data/generate_data.py`).

In [ ]:
raw = load_raw()
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
# Target balance in the raw data
raw[TARGET].value_counts(normalize=True).rename("proportion").to_frame()

## 2. Exploratory Data Analysis

### 2.1 Class distribution

Churn is the minority class — this imbalance motivates SMOTE later.

In [ ]:
df_eda = clean(raw)   # 0/1 target, numeric TotalCharges
fig, ax = plt.subplots(figsize=(6,4))
counts = df_eda[TARGET].value_counts().sort_index()
ax.bar(["Retained (0)","Churned (1)"], counts.values, color=["#4C72B0","#C44E52"])
for i,v in enumerate(counts.values):
    ax.text(i, v, f"{v}\n({v/len(df_eda):.1%})", ha="center", va="bottom")
ax.set_title("Class Distribution (Churn)"); ax.set_ylabel("Customers")
plt.show()

### 2.2 Numeric feature distributions by churn

Short-tenure and high-monthly-charge customers churn more.

In [ ]:
num_cols = ["tenure","MonthlyCharges","TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(16,4))
for ax,c in zip(axes, num_cols):
    sns.histplot(data=df_eda, x=c, hue=TARGET, bins=30, kde=True, ax=ax,
                 palette={0:"#4C72B0",1:"#C44E52"}, alpha=0.6)
    ax.set_title(f"Distribution of {c}")
plt.tight_layout(); plt.show()

### 2.3 Categorical churn rates

Contract type and internet service are strong churn signals.

In [ ]:
cat_cols = ["Contract","InternetService","PaymentMethod","TechSupport"]
fig, axes = plt.subplots(2, 2, figsize=(14,9))
for ax, c in zip(axes.ravel(), cat_cols):
    (df_eda.groupby(c)[TARGET].mean().sort_values()
        .plot(kind="bar", ax=ax, color="#C44E52"))
    ax.set_title(f"Churn rate by {c}"); ax.set_ylabel("churn rate")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### 2.4 Correlation heatmap (with engineered features)

Computed after feature engineering so the new numeric features are included.

In [ ]:
df_fe = add_features(df_eda)
print("After feature engineering:", df_fe.shape, "->", df_fe.shape[1]-1, "features")
corr = df_fe.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(16,13))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink":0.6})
ax.set_title("Correlation Heatmap (numeric + engineered features)")
plt.show()

## 3. Feature engineering (25+ features)

All transforms are stateless (row-wise) so the identical function powers both training and the FastAPI service.

In [ ]:
engineered = sorted(set(df_fe.columns) - set(df_eda.columns))
print(f"{len(engineered)} engineered features:")
for f in engineered:
    print("  -", f)

In [ ]:
# A peek at a few engineered signals most correlated with churn
top_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(12)
top_corr.to_frame("abs_corr_with_churn")

## 4. Train/test split + preprocessing pipeline

Standardize numerics, one-hot encode categoricals. SMOTE is placed *inside* the pipeline so it only ever sees training folds — no leakage into validation.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from src.train import build_preprocessor, candidate_models, make_pipeline

y = df_fe[TARGET]
X = df_fe.drop(columns=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CFG["data"]["test_size"], stratify=y, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)

preprocessor = build_preprocessor(X)
cv = StratifiedKFold(n_splits=CFG["training"]["cv_folds"], shuffle=True, random_state=SEED)

### 4.1 SMOTE effect on the training class balance

In [ ]:
from imblearn.over_sampling import SMOTE
Xt = preprocessor.fit_transform(X_train)
Xs, ys = SMOTE(random_state=SEED).fit_resample(Xt, y_train)
print("Before SMOTE:", dict(pd.Series(y_train).value_counts().sort_index()))
print("After  SMOTE:", dict(pd.Series(ys).value_counts().sort_index()))

## 5. Compare 5 models with 5-fold cross-validation

Each model is wrapped in `preprocess → SMOTE → classifier` and scored by ROC-AUC.

In [ ]:
results = []
for name, model in candidate_models().items():
    pipe = make_pipeline(preprocessor, model, use_smote=True)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    results.append({"model": name, "cv_roc_auc": scores.mean(), "cv_std": scores.std()})
    print(f"{name:<20} ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")

results_df = pd.DataFrame(results).sort_values("cv_roc_auc", ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
r = results_df.sort_values("cv_roc_auc")
ax.barh(r["model"], r["cv_roc_auc"], xerr=r["cv_std"], color="#4C72B0", capsize=4)
ax.set_xlim(0.5,1.0); ax.set_xlabel("Cross-validated ROC-AUC")
ax.set_title("5-Model Comparison (5-fold CV)")
for i,(v,s) in enumerate(zip(r["cv_roc_auc"], r["cv_std"])): ax.text(v, i, f" {v:.3f}", va="center")
plt.show()

**XGBoost** comes out on top — it captures the non-linear interaction effects that the linear models cannot.

## 6. Hyperparameter tuning — GridSearchCV on XGBoost

In [ ]:
from xgboost import XGBClassifier
xgb_pipe = make_pipeline(preprocessor,
    XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1), use_smote=True)
param_grid = {
    "model__n_estimators": [300],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.9],
    "model__colsample_bytree": [0.9],
}
grid = GridSearchCV(xgb_pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")
best = grid.best_estimator_

## 7. Evaluate the tuned model on the hold-out test set

In [ ]:
from src.evaluate import compute_metrics
from sklearn.metrics import classification_report

y_pred  = best.predict(X_test)
y_proba = best.predict_proba(X_test)[:, 1]
metrics = compute_metrics(y_test, y_pred, y_proba)
print("Hold-out test metrics:")
for k, v in metrics.items():
    print(f"  {k:<12}: {v:.4f}")
print("\n", classification_report(y_test, y_pred, target_names=["Retained","Churned"]))

### 7.1 Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Retained","Churned"], yticklabels=["Retained","Churned"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix — XGBoost")
plt.show()

### 7.2 ROC curve

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, _ = roc_curve(y_test, y_proba); auc = roc_auc_score(y_test, y_proba)
fig, ax = plt.subplots(figsize=(5.5,4.5))
ax.plot(fpr, tpr, color="#C44E52", lw=2, label=f"XGBoost (AUC = {auc:.3f})")
ax.plot([0,1],[0,1],"--",color="gray",label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve"); ax.legend(loc="lower right")
plt.show()

### 7.3 Feature importance

In [ ]:
importances = best.named_steps["model"].feature_importances_
feat_names  = best.named_steps["prep"].get_feature_names_out()
order = np.argsort(importances)[::-1][:20]
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(np.array(feat_names)[order][::-1], importances[order][::-1], color="#55A868")
ax.set_title("Top 20 Feature Importances — XGBoost"); ax.set_xlabel("Importance (gain)")
plt.show()

## 8. Experiment tracking with MLflow

Log the tuned model's parameters and hold-out metrics. Browse runs with `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", CFG["mlflow"]["tracking_uri"]))
    mlflow.set_experiment(CFG["mlflow"]["experiment_name"])
    with mlflow.start_run(run_name="xgboost_notebook"):
        mlflow.log_params({k.replace("model__",""):v for k,v in grid.best_params_.items()})
        mlflow.log_param("smote", True)
        mlflow.log_metrics(metrics)
        mlflow.log_metric("best_cv_roc_auc", float(grid.best_score_))
    print("Logged run to MLflow.")
except Exception as e:
    print("MLflow logging skipped:", e)

## 9. Persist the model & save a prediction example

In [ ]:
import joblib
from pathlib import Path
Path("../models").mkdir(exist_ok=True)
joblib.dump({"pipeline": best, "feature_columns": X.columns.tolist(),
             "raw_input_columns": [c for c in raw.columns if c not in (TARGET,'customerID')]},
            "../models/churn_xgb_pipeline.joblib")
print("Saved -> models/churn_xgb_pipeline.joblib")
print("Serve it with:  uvicorn api.main:app --port 8000   (then POST to /predict)")

## Summary

| Step | Result |
|------|--------|
| Dataset | ~7,000 customers, 53 features after engineering |
| Best model | **XGBoost** (SMOTE + tuned via GridSearchCV) |
| Test accuracy | **~0.91** |
| Test ROC-AUC | **~0.88** |
| Serving | FastAPI `/predict` endpoint, Dockerized |
| Tracking | MLflow experiment `customer-churn-prediction` |

The strongest churn drivers are month-to-month contracts, fiber-optic internet, short tenure,
electronic-check payment, and the absence of tech-support / online-security add-ons.